# USD/JPY Market Analysis

This notebook creates ten-year and twelve-month views of USD/JPY using the Federal Reserve Bank of St. Louis series `DEXJPUS`.

The exchange rate is quoted as Japanese yen per U.S. dollar. A rising series therefore indicates yen depreciation, while a falling series indicates yen appreciation.


## 1. Environment and paths

Run this cell first. Generated figures are saved in the project’s `outputs/` directory.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import FuncFormatter


def find_project_root(start: Path) -> Path:
    """Locate the yen project from the repository root or notebook folder."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if candidate.name == "yen-intervention-article":
            return candidate
        nested = candidate / "yen-intervention-article"
        if nested.is_dir():
            return nested
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Charts save to: {OUTPUT_DIR}")


## 2. Ten-year context

Place the latest USD/JPY move within a longer monetary-policy and exchange-rate cycle.


In [ ]:
# Set the date range to the most recent 10 years
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=10)

# Download USD/JPY data from FRED
url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    f"?id=DEXJPUS&cosd={start_date:%Y-%m-%d}&coed={end_date:%Y-%m-%d}"
)

usd_jpy = pd.read_csv(url)

# FRED may call the first column DATE or observation_date,
# so rename it based on its position
usd_jpy = usd_jpy.rename(
    columns={
        usd_jpy.columns[0]: "Date",
        "DEXJPUS": "USD_JPY"
    }
)

# Clean the data
usd_jpy["Date"] = pd.to_datetime(usd_jpy["Date"])
usd_jpy["USD_JPY"] = pd.to_numeric(
    usd_jpy["USD_JPY"],
    errors="coerce"
)

usd_jpy = (
    usd_jpy
    .dropna(subset=["USD_JPY"])
    .sort_values("Date")
)

# Obtain the latest observation
latest_date = usd_jpy["Date"].iloc[-1]
latest_rate = usd_jpy["USD_JPY"].iloc[-1]

# Create the graph
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    usd_jpy["Date"],
    usd_jpy["USD_JPY"],
    color="#1f4e79",
    linewidth=2
)

ax.fill_between(
    usd_jpy["Date"],
    usd_jpy["USD_JPY"],
    usd_jpy["USD_JPY"].min(),
    color="#1f4e79",
    alpha=0.08
)

# Mark the latest exchange rate
ax.scatter(
    latest_date,
    latest_rate,
    color="#c0392b",
    s=55,
    zorder=3
)

ax.annotate(
    f"Latest: ¥{latest_rate:.2f}",
    xy=(latest_date, latest_rate),
    xytext=(-105, 20),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#c0392b")
)

# Formatting
ax.set_title(
    "USD/JPY Exchange Rate — Last 10 Years",
    fontsize=18,
    fontweight="bold",
    loc="left"
)

ax.set_xlabel("")
ax.set_ylabel("Japanese Yen per U.S. Dollar", fontsize=11)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda value, position: f"¥{value:.0f}")
)

ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.text(
    0.125,
    0.02,
    f"Source: Federal Reserve Bank of St. Louis, DEXJPUS | "
    f"Latest observation: {latest_date:%B %d, %Y}",
    fontsize=9,
    color="dimgray"
)

plt.tight_layout(rect=[0, 0.05, 1, 1])

fig.savefig(
    OUTPUT_DIR / "usd_jpy_10_years.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

## 3. Twelve-month view

Focus on the most recent year, where intervention expectations and rapid position unwinds are easier to see.


In [ ]:
# Set date range to the latest 12 months
end_date = pd.Timestamp.today().normalize()
start_date = end_date - pd.DateOffset(years=1)

# Download USD/JPY exchange-rate data from FRED
url = (
    "https://fred.stlouisfed.org/graph/fredgraph.csv"
    f"?id=DEXJPUS&cosd={start_date:%Y-%m-%d}&coed={end_date:%Y-%m-%d}"
)

usd_jpy = pd.read_csv(url)

# Rename columns safely
usd_jpy = usd_jpy.rename(
    columns={
        usd_jpy.columns[0]: "Date",
        "DEXJPUS": "USD_JPY"
    }
)

# Clean the data
usd_jpy["Date"] = pd.to_datetime(usd_jpy["Date"])
usd_jpy["USD_JPY"] = pd.to_numeric(
    usd_jpy["USD_JPY"],
    errors="coerce"
)

usd_jpy = (
    usd_jpy
    .dropna(subset=["USD_JPY"])
    .sort_values("Date")
)

# Get the latest observation
latest_date = usd_jpy["Date"].iloc[-1]
latest_rate = usd_jpy["USD_JPY"].iloc[-1]

# Create the graph
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(
    usd_jpy["Date"],
    usd_jpy["USD_JPY"],
    color="#1f4e79",
    linewidth=2.2
)

ax.fill_between(
    usd_jpy["Date"],
    usd_jpy["USD_JPY"],
    usd_jpy["USD_JPY"].min(),
    color="#1f4e79",
    alpha=0.08
)

# Mark the latest rate
ax.scatter(
    latest_date,
    latest_rate,
    color="#c0392b",
    s=60,
    zorder=3
)

ax.annotate(
    f"Latest: ¥{latest_rate:.2f}",
    xy=(latest_date, latest_rate),
    xytext=(-110, 25),
    textcoords="offset points",
    fontsize=10,
    fontweight="bold",
    arrowprops=dict(arrowstyle="->", color="#c0392b")
)

# Format the chart
ax.set_title(
    "USD/JPY Exchange Rate — Last 12 Months",
    fontsize=18,
    fontweight="bold",
    loc="left"
)

ax.set_xlabel("")
ax.set_ylabel("Japanese Yen per U.S. Dollar", fontsize=11)

ax.yaxis.set_major_formatter(
    FuncFormatter(lambda value, position: f"¥{value:.0f}")
)

ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.grid(axis="x", visible=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.text(
    0.125,
    0.02,
    f"Source: Federal Reserve Bank of St. Louis, DEXJPUS | "
    f"Latest observation: {latest_date:%B %d, %Y}",
    fontsize=9,
    color="dimgray"
)

plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

# Save as a high-resolution image
fig.savefig(
    OUTPUT_DIR / "usd_jpy_last_12_months.png",
    dpi=300,
    bbox_inches="tight"
)

## Limitations

The chart documents exchange-rate movement but does not independently identify intervention. Establishing causation requires official Ministry of Finance intervention data, policy announcements, positioning evidence, and contemporaneous market reporting. Data are downloaded through the current date, so future reruns may differ from the committed figure.
